<a href="https://colab.research.google.com/github/remythomasvo/FinLLM/blob/main/notebooks/01_market_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip install -q yfinance

In [34]:
import yfinance as yf
import pandas as pd
import numpy as np
# Download historical NVIDIA prices
nvda = yf.download(
    "NVDA",
    start="2016-01-01",
    end="2026-11-10",
    auto_adjust=True,
    progress=False
)

# Download historical S&P 500 ETF prices
spy = yf.download(
    "SPY",
    start="2016-01-01",
    end="2026-11-10",
    auto_adjust=True,
    progress=False
)

# Simplify column names if yfinance uses a MultiIndex
for df in (nvda, spy):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

print("NVIDIA:")
print(nvda.head())

print("\nSPY:")
print(spy.head())


NVIDIA:
Price          Close      High       Low      Open     Volume
Date                                                         
2016-01-04  0.787745  0.792855  0.779714  0.785798  358076000
2016-01-05  0.800399  0.813784  0.790908  0.802589  490272000
2016-01-06  0.767303  0.790908  0.758298  0.787258  449344000
2016-01-07  0.736883  0.753188  0.727149  0.748077  645304000
2016-01-08  0.721065  0.747104  0.719605  0.746374  398472000

SPY:
Price            Close        High         Low        Open     Volume
Date                                                                 
2016-01-04  169.051788  169.060193  167.008225  168.606075  222353500
2016-01-05  169.337692  169.791810  168.236024  169.371325  110845800
2016-01-06  167.201599  168.244393  166.175615  166.797924  152112600
2016-01-07  163.190201  166.041088  162.803349  164.266641  213436100
2016-01-08  161.398987  164.704006  161.113060  164.148963  209817200


In [35]:
print(nvda.shape)
print(nvda.info())
print(nvda.describe())
print(nvda.isnull().sum())

(2729, 5)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2729 entries, 2016-01-04 to 2026-11-09
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   2729 non-null   float64
 1   High    2729 non-null   float64
 2   Low     2729 non-null   float64
 3   Open    2729 non-null   float64
 4   Volume  2729 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 127.9 KB
None
Price        Close         High          Low         Open        Volume
count  2729.000000  2729.000000  2729.000000  2729.000000  2.729000e+03
mean     49.122724    49.930152    48.287868    49.157595  4.345810e+08
std      66.729932    67.758662    65.739398    66.834110  2.610528e+08
min       0.613745     0.622019     0.602307     0.603037  6.552850e+07
25%       4.727848     4.790244     4.673377     4.748069  2.579960e+08
50%      14.190047    14.360301    13.884249    14.145931  3.912200e+08
75%      73.686432    74.395362    72.291461  

In [36]:
nvda.to_csv("nvda_prices.csv")
spy.to_csv("spy_prices.csv")

print("Files saved successfully!")

Files saved successfully!


In [37]:
from google.colab import files

files.download("nvda_prices.csv")
files.download("spy_prices.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Feature Engineering
Calculating daily returns, five-day momentum, 20-day rolling volatility, and relative volume using pandas.

In [38]:
nvda["return_1day"] = nvda["Close"].pct_change() # calculate daily returns
nvda["return_5day"] = nvda['Close'].pct_change(5) # calculate returns over past 5 days
nvda["volatility_20d"] = nvda["return_1day"].rolling(20).std() # calculate 20 day rolling volatility
nvda["rel_vol"] = (nvda["Volume"] / nvda["Volume"].rolling(20).mean()) # calculate relative volume


print(nvda[["return_1day", "return_5day", "volatility_20d", "rel_vol"]].head(25)) #display info

Price       return_1day  return_5day  volatility_20d   rel_vol
Date                                                          
2016-01-04          NaN          NaN             NaN       NaN
2016-01-05     0.016064          NaN             NaN       NaN
2016-01-06    -0.041350          NaN             NaN       NaN
2016-01-07    -0.039645          NaN             NaN       NaN
2016-01-08    -0.021467          NaN             NaN       NaN
2016-01-11     0.001688    -0.083102             NaN       NaN
2016-01-12     0.016846    -0.082396             NaN       NaN
2016-01-13    -0.030484    -0.071995             NaN       NaN
2016-01-14    -0.020164    -0.053171             NaN       NaN
2016-01-15    -0.054412    -0.085049             NaN       NaN
2016-01-19     0.008115    -0.079178             NaN       NaN
2016-01-20     0.005488    -0.089463             NaN       NaN
2016-01-21     0.011645    -0.049898             NaN       NaN
2016-01-22     0.023381    -0.007673             NaN   

In [39]:
nvda["NextDayClose"] = nvda["Close"].shift(-1) # checks price at close of next day
nvda["target"] = (nvda["NextDayClose"] > nvda["Close"]).astype(int) # compare tomorrow's close with todays

nvda.loc[nvda["NextDayClose"].isna(), "target"] = float("nan") # final day has no NextDayClose
print(nvda[["Close", "NextDayClose", "target"]].tail(10)) #check results

Price            Close  NextDayClose  target
Date                                        
2026-10-27  217.940002    215.380005     0.0
2026-10-28  215.380005    213.660004     0.0
2026-10-29  213.660004    204.639999     0.0
2026-10-30  204.639999    202.059998     0.0
2026-11-02  202.059998    206.690002     1.0
2026-11-03  206.690002    200.130005     0.0
2026-11-04  200.130005    181.600006     0.0
2026-11-05  181.600006    186.639999     1.0
2026-11-06  186.639999    178.669998     0.0
2026-11-09  178.669998           NaN     NaN


In [43]:
features = ["return_1day", "return_5day", "volatility_20d", "rel_vol"] # define columns used for predictions

model_data = nvda.dropna(subset = features + ["target"]).copy() # create a clean dataset

X = model_data[features] # Separate inputs from target
y = model_data["target"].astype(int)

print("Inputs: ", X.shape) # Check results
print("Outputs: ", y.shape)
print(y.value_counts())

Inputs:  (2708, 4)
Outputs:  (2708,)
target
1    1470
0    1238
Name: count, dtype: int64


In [44]:
print(model_data[["Close", "NextDayClose", "target"]].tail())
print(model_data[features + ["target"]].isna().sum())

Price            Close  NextDayClose  target
Date                                        
2026-11-02  202.059998    206.690002     1.0
2026-11-03  206.690002    200.130005     0.0
2026-11-04  200.130005    181.600006     0.0
2026-11-05  181.600006    186.639999     1.0
2026-11-06  186.639999    178.669998     0.0
Price
return_1day       0
return_5day       0
volatility_20d    0
rel_vol           0
target            0
dtype: int64


In [45]:
# Find the point where we split the dataset
split = int(len(X) * 0.8)

# Split the input features
X_train = X.iloc[:split]
X_test = X.iloc[split:]

# Split the target values
Y_train = Y.iloc[:split]
Y_test = Y.iloc[split:]

# Check the sizes
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 2166
Testing rows: 542


In [48]:
model_data.to_csv("nvda_features.csv", index=True)